## 데이터 및 라이브러리 로딩

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn import set_config
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

# 한글 윈도우에서 모델 출력(HTML) 시 UnicodeDecodeError 나는 것 방지
set_config(display='text')

# 데이터 폴더 경로: 노트북과 같은 폴더면 '.' 그대로
# 코랩에서 드라이브에 둔 경우 예) DATA_DIR = '/content/drive/MyDrive/ESAA/미니프로젝트3'
DATA_DIR = '.'

# 팀 공통 설정 (모두 같은 값 사용)
SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

train = pd.read_csv(os.path.join(DATA_DIR, 'movies_train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'movies_test.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))

## 데이터탐색

In [ ]:
train.info()

In [ ]:
train.head()

In [ ]:
test.head()

## 데이터 전처리

In [ ]:
# 결측치가 많은 데이터 제거
train = train.drop(['dir_prev_bfnum'], axis=1)
test = test.drop(['dir_prev_bfnum'], axis=1)

In [ ]:
# 감독명 : 너무 다양해서 제거
train = train.drop(['director'], axis=1)
test = test.drop(['director'], axis=1)

In [ ]:
# 제목 : 의미가 없기 때문에 제거
train = train.drop(['title'], axis=1)
test = test.drop(['title'], axis=1)

In [ ]:
train.distributor.value_counts()

In [ ]:
# 상위 5개의 배급사를 제외하고 '기타'로 처리
top5 = train['distributor'].value_counts().index[:5]

train['distributor'] = train['distributor'].where(train['distributor'].isin(top5), '기타')
test['distributor'] = test['distributor'].where(test['distributor'].isin(top5), '기타')

In [ ]:
# 개봉일을 바탕으로 년, 월 변수 생성
for df in (train, test):
    date = pd.to_datetime(df['release_time'])
    df['년'] = date.dt.year
    df['월'] = date.dt.month

train = train.drop(['release_time'], axis=1)
test = test.drop(['release_time'], axis=1)

In [ ]:
# 원핫 인코딩
train = pd.get_dummies(train)
test = pd.get_dummies(test)

## 모델정의 및 학습

In [ ]:
train_x = train.drop(['box_off_num'], axis=1)
train_y = train['box_off_num']

# test 컬럼을 train과 같은 순서로 맞춤 (원핫 후 한쪽에만 있는 컬럼은 0으로 채움)
test = test.reindex(columns=train_x.columns, fill_value=0)

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=SEED)

# 팀 공통 검증: 5-Fold RMSE (작을수록 좋음)
scores = cross_val_score(model, train_x, train_y, cv=kf, scoring='neg_root_mean_squared_error')
print(f'CV RMSE: {-scores.mean():,.0f} (± {scores.std():,.0f})')

model.fit(train_x, train_y);

## 학습된 모델로 예측 데이터 생성

In [ ]:
pred = model.predict(test)

## 제출파일 생성

In [ ]:
submission['box_off_num'] = pred
submission.to_csv('베이스라인.csv', index=False)
submission.head()